# Stackelberg : la performativité sans mystère
**Navigation** : [<< 9-BackwardInduction](GameTheory-09-BackwardInduction-Python.ipynb) | [Index](README.md) | [9c-Stackelberg-SecurityGame >>](GameTheory-09c-Stackelberg-SecurityGame-Python.ipynb) | [Jumeau C#](GameTheory-09b-Commitment-Stackelberg-CSharp.ipynb)


Une action **inférieure contre une réponse fixée** dans le jeu simultané peut devenir la meilleure si l'agent peut **s'y engager** avant que l'autre ne choisisse — parce que l'engagement **transforme la meilleure réponse de l'autre**. C'est le cœur de la solution de Stackelberg [von Stackelberg, 1934], et c'est aussi, déguisé en théorie des jeux, un noyau mathématique propre pour ce qu'Austin [1962] appelle le **performatif** : un énoncé (ou un acte) qui ne décrit pas le monde mais le *fait*.

```text
sigma_L  ->  BR_F(sigma_L)  ->  u_L(sigma_L, BR_F(sigma_L))
```

> **L'acte n'a plus seulement une valeur par son RÉSULTAT DIRECT. Il a une valeur parce qu'il MODIFIE L'ESPACE DE RÉPONSE D'AUTRUI.**

La condition de félicité d'Austin — ce qui distingue une annonce qui *fait* d'une annonce qui ne *dit* rien — se traduit ici en termes brutaux : **l'engagement doit être crédible**. S'il ne l'est pas, rien ne change. S'il l'est, l'annonce devient **causalement constitutive** du jeu qui suit :

```text
representation publique  ->  anticipation collective  ->  changement des actions  ->  nouvelle realite strategique
```

Ce notebook démonte la mécanique sur un jeu d'entrée sur marché, en quatre régimes calculés exactement : le jeu simultané, la mise séquentielle sans engagement, l'engagement contraignant, et l'engagement révocable — puis il **mesure** ce que coûte la crédibilité, et montre que le seuil à payer est exactement l'écart de tentation du déviateur. Une performativité sans mystère : chaque affirmation de ce notebook est un nombre calculé dans une cellule.

Le fil suivra la série [GameTheory](GameTheory-02-NormalForm-Python.ipynb) (jeux sous forme normale, [équilibres de Nash](GameTheory-04-NashEquilibrium-Python.ipynb), [induction à rebours](GameTheory-09-BackwardInduction-Python.ipynb)) et rejoint le chantier « strate 7 » (vocabulaires et extensions) : l'engagement est l'archétype de l'acte qui change l'espace des actions disponibles plutôt que d'en choisir une.

**Objectifs** : calculer meilleures réponses et engagements, distinguer annonce et lien, vérifier un certificat de retenue locale. **Prérequis** : formes normales et induction à rebours. **Durée estimée** : 60–90 minutes.

Python standard uniquement ; les tableaux et arbres sont calculés depuis les matrices.


In [1]:
import math
# === Le jeu d'entree sur marche : l'entrant (E) face a l'incumbent (I) ===
# Convention : chaque case = (u_entrant, u_incumbent).
#   Entrant : 'In' (entrer) / 'Out' (rester dehors).
#   Incumbent : 'Acc' (accommoder, se partager le marche) / 'Fight' (guerre de prix).
ACTIONS_E = ['In', 'Out']
ACTIONS_I = ['Acc', 'Fight']
U = {('In',  'Acc'):   (2,  1),   # duopole accommodant
     ('In',  'Fight'): (-1, -2),  # guerre de prix : les deux perdent
     ('Out', 'Acc'):   (0,  0),   # marche vide, rien a accommoder
     ('Out', 'Fight'): (0,  3)}   # dissuasion reussie : monopole intact

def u_E(ae, ai): return U[(ae, ai)][0]
def u_I(ae, ai): return U[(ae, ai)][1]

def br_E(ai):  # meilleure reponse de l'entrant a l'action de l'incumbent
    return max(ACTIONS_E, key=lambda ae: u_E(ae, ai))

def br_I(ae):  # meilleure reponse de l'incumbent a l'action de l'entrant
    return max(ACTIONS_I, key=lambda ai: u_I(ae, ai))

print('Matrice de gains (u_entrant, u_incumbent) :')
print(f'{"":10} {"Acc":>12} {"Fight":>12}')
for ae in ACTIONS_E:
    row = ' '.join(f'{str(U[(ae, ai)]):>12}' for ai in ACTIONS_I)
    print(f'{ae:10} {row}')
print()
print("Meilleures reponses :")
for ai in ACTIONS_I:
    print(f'  BR_entrant({ai:5}) = {br_E(ai):4}   (gains E : In={u_E("In", ai):>2}, Out={u_E("Out", ai):>2})')
for ae in ACTIONS_E:
    print(f'  BR_incumbent({ae:4}) = {br_I(ae):5}   (gains I : Acc={u_I(ae, "Acc"):>2}, Fight={u_I(ae, "Fight"):>2})')

Matrice de gains (u_entrant, u_incumbent) :
                    Acc        Fight
In               (2, 1)     (-1, -2)
Out              (0, 0)       (0, 3)

Meilleures reponses :
  BR_entrant(Acc  ) = In     (gains E : In= 2, Out= 0)
  BR_entrant(Fight) = Out    (gains E : In=-1, Out= 0)
  BR_incumbent(In  ) = Acc     (gains I : Acc= 1, Fight=-2)
  BR_incumbent(Out ) = Fight   (gains I : Acc= 0, Fight= 3)


### Lecture : la valeur de Fight vit dans la ligne Out

Regardez les deux meilleures réponses de l'incumbent : contre une **entrée**, il accommode (1 > −2) ; contre l'**abstention**, il « combat » (3 > 0) — combattre un marché vide ne coûte rien et laisse le monopole intact. L'action `Fight` n'est donc pas dominée dans l'absolu, mais elle est **strictement inférieure contre cette entrée** : sur la ligne `In`, elle vaut −2 contre 1 pour `Acc`. Toute sa valeur (3) vit dans la ligne `Out` — une colonie de gains qui n'existe que si l'entrant reste dehors.

C'est la tension de tout le notebook : `Fight` est l'action qu'il *faudrait* pouvoir choisir, précisément dans le monde où elle ne coûte rien — et l'entrant le sait. La question de Stackelberg est de savoir si l'incumbent peut **faire exister** la ligne `Out`. La question d'Austin est de savoir à quel prix l'annonce « je combattrai » devient un fait plutôt qu'un souffle.

In [2]:
# === Regime 1 : le jeu simultane -- equilibres de Nash en actions pures ===
nes = [(ae, ai) for ae in ACTIONS_E for ai in ACTIONS_I
       if ae == br_E(ai) and ai == br_I(ae)]
print('Equilibres de Nash (actions pures) du jeu simultane :')
for ae, ai in nes:
    print(f'  ({ae}, {ai})  ->  u_E = {u_E(ae, ai):>2}, u_I = {u_I(ae, ai):>2}')

# === Regime 2 : mise sequentielle SANS engagement (l'entrant joue le premier) ===
# Induction a rebours : l'incumbent repond, l'entrant anticipe la reponse.
print()
print('Mise sequentielle (entrant puis incumbent), induction a rebours :')
for ae in ACTIONS_E:
    print(f'  si E joue {ae:4} : I repond {br_I(ae):5} -> u_E = {u_E(ae, br_I(ae)):>2}')
seq_ae = max(ACTIONS_E, key=lambda ae: u_E(ae, br_I(ae)))
print(f'  -> issue sequentielle : ({seq_ae}, {br_I(seq_ae)}), u_E = {u_E(seq_ae, br_I(seq_ae))}, u_I = {u_I(seq_ae, br_I(seq_ae))}')

Equilibres de Nash (actions pures) du jeu simultane :
  (In, Acc)  ->  u_E =  2, u_I =  1
  (Out, Fight)  ->  u_E =  0, u_I =  3

Mise sequentielle (entrant puis incumbent), induction a rebours :
  si E joue In   : I repond Acc   -> u_E =  2
  si E joue Out  : I repond Fight -> u_E =  0
  -> issue sequentielle : (In, Acc), u_E = 2, u_I = 1


### Lecture : deux équilibres simultanés, un seul survit à la séquentialisation

Le jeu simultané porte **deux** équilibres de Nash : `(In, Acc)` où l'entrant vient et se fait accueillir (l'incumbent touche 1), et `(Out, Fight)` où l'entrant reste dehors face à la menace (l'incumbent touche 3). Le second vaut trois fois le premier pour l'incumbent — mais regardez ce que la mise séquentielle en fait : dès que l'entrant peut *tester* la menace en jouant le premier, l'induction à rebours la dissout. L'incumbent qui fait face à une entrée réelle accommode (1 > −2), l'entrant le sait, entre, et l'issue retombe sur `(In, Acc)` : **u_I = 1**.

L'équilibre `(Out, Fight)` n'était pas faux — c'est un équilibre de Nash parfaitement valide de la forme normale — il était **incroyable** : sa valeur repose sur une action (`Fight` face à `In`) que l'incumbent ne choisirait jamais au moment de la choisir. La différence 3 − 1 = **2** n'est pas perdue : elle est *en attente*. C'est exactement le montant qu'un engagement contraignant peut débloquer dans cette comparaison de mécanismes.

In [3]:
# === Regime 3 : l'engagement CONTRAIGNANT (l'incumbent joue le premier, de facon liante) ===
# Pour chaque action a laquelle l'incumbent peut s'engager, l'entrant best-repond, l'incumbent encaisse.
print('Engagement contraignant : tableau (engagement -> reponse de E -> gain de I)')
commit = {}
for ai in ACTIONS_I:
    r = br_E(ai)
    commit[ai] = u_I(r, ai)
    print(f'  s engager a {ai:5} -> E repond {r:4} -> u_I = {u_I(r, ai):>2}')
best_ai = max(ACTIONS_I, key=lambda ai: commit[ai])
gain_seq = u_I(seq_ae, br_I(seq_ae))
print(f'  -> engagement optimal : {best_ai} (u_I = {commit[best_ai]}), contre {gain_seq} sans engagement : delta = {commit[best_ai] - gain_seq:+d}')

# === Engagement MIXTE : l'incumbent s'engage a une probabilite p de Fight ===
print()
print('Engagement mixte (p = P(Fight)) : meilleure reponse de E et gain de I')
print(f'{"p":>5} {"BR_E":>6} {"u_I":>8}')
for k in range(21):
    p = k / 20
    in_pay = 2 - 3 * p        # u_E(In) si I mixe (p Fight, 1-p Acc)
    r = 'In' if in_pay > 0 else ('Out' if in_pay < 0 else 'indifferent')
    if r == 'In':
        val = 1 - 3 * p       # u_I si E entre
    else:
        val = 3 * p           # u_I si E reste dehors (cas favorable a l indifferenciation ; le pire cas pour I serait 1 - 3*p)
    print(f'{p:>5.2f} {r:>10} {val:>8.2f}')
from fractions import Fraction
seuil_p = Fraction(2,3)
print(f'Au seuil exact p={seuil_p} : BR_E=In ou Out ; u_I favorable={3*seuil_p}, défavorable={1-3*seuil_p}')


Engagement contraignant : tableau (engagement -> reponse de E -> gain de I)
  s engager a Acc   -> E repond In   -> u_I =  1
  s engager a Fight -> E repond Out  -> u_I =  3
  -> engagement optimal : Fight (u_I = 3), contre 1 sans engagement : delta = +2

Engagement mixte (p = P(Fight)) : meilleure reponse de E et gain de I
    p   BR_E      u_I
 0.00         In     1.00
 0.05         In     0.85
 0.10         In     0.70
 0.15         In     0.55
 0.20         In     0.40
 0.25         In     0.25
 0.30         In     0.10
 0.35         In    -0.05
 0.40         In    -0.20
 0.45         In    -0.35
 0.50         In    -0.50
 0.55         In    -0.65
 0.60         In    -0.80
 0.65         In    -0.95
 0.70        Out     2.10
 0.75        Out     2.25
 0.80        Out     2.40
 0.85        Out     2.55
 0.90        Out     2.70
 0.95        Out     2.85
 1.00        Out     3.00
Au seuil exact p=2/3 : BR_E=In ou Out ; u_I favorable=2, défavorable=-1


### Lecture : l'action sous-optimale devient l'engagement unique optimal

Le tableau de l'engagement est la cellule centrale du notebook :

| engagement de I | réponse de E | gain de I |
|---|---|---|
| `Acc` | `In` | 1 |
| **`Fight`** | **`Out`** | **3** |

S'engager à `Acc` ne change rien (l'entrant vient, gain 1). S'engager à `Fight` **transforme la meilleure réponse de l'entrant** — face à un adversaire lié à la guerre de prix, `Out` (0) domine `In` (−1) — et rapporte 3. Le delta est **+2**, exactement l'écart entre les deux équilibres du jeu simultané : l'engagement n'invente pas de valeur, il **sélectionne** l'équilibre que la menace incroyable ne pouvait pas tenir.

Et le point décisif, celui qui donne son titre à l'issue : `Fight` était **strictement inférieure contre une entrée** (−2 < 1 sur la ligne `In`). Elle devient l'engagement *unique* optimal — non parce que sa valeur directe aurait changé (combattre une entrée vaut toujours −2), mais parce que **commettre la transforme en device de sélection** : l'acte ne vaut pas par son résultat direct, il vaut par la modification de l'espace de réponse d'autrui. C'est le sens exact de la chaîne `sigma_L -> BR_F(sigma_L) -> u_L` ouverte en tête de notebook.

L'engagement **mixte** referme la porte d'une évasion : aucun mixage ne bat le sommet pur. Tant que p < 2/3, l'entrant entre et le gain de l'incumbent décroît (1 − 3p) ; pour p > 2/3, l'entrant sort et le gain plafonne à 3p, maximal à p = 1. À p = 2/3, les deux réponses de l’entrant sont optimales : la sélection favorable au leader donne 2, la défavorable −1. Le balayage au pas 0,05 ne contient pas ce seuil ; il est calculé séparément. La valeur d'engagement de ce jeu est un **sommet du simplexe**, pas un intérieur — la nuance honnête étant qu'il existe d'autres jeux (les jeux d'inspection, où le contrôleur doit rester imprévisible) où c'est l'intérieur qui gagne : le phénomène « mixer bat commettre » existe, ce n'est simplement pas celui d'ici.

La sélection d’un Nash existant est un résultat de cette matrice ; elle ne caractérise pas tous les jeux d’engagement. Le duel publicitaire ci-dessous présente déjà un meilleur engagement mixte intérieur.


<a id="duel-publicitaire"></a>

## Exercice 1 : le duel publicitaire — équilibres et action candidate

Le jeu de ce notebook n'est pas le seul où vit un engagement. Soit le duel publicitaire suivant : deux firmes A (leader potentiel) et B ; A choisit `PasCampagne` / `Campagne`, B choisit `Reste` / `Attaque`. Gains `(u_A, u_B)` : `('PasCampagne','Reste'): (3, 3)`, `('PasCampagne','Attaque'): (1, 4)`, `('Campagne','Reste'): (2, 0)`, `('Campagne','Attaque'): (0, -1)`.

Calculez les meilleures réponses, listez **tous** les équilibres de Nash en actions pures du jeu simultané, puis l'issue de la mise séquentielle où B joue le premier. Identifiez enfin l'action de A qui est sous-optimale conditionnellement à `Attaque` — la candidate au rôle que `Fight` joue ci-dessus.

La fonction étudiante reçoit la matrice. Retourner les correspondances complètes `br_a`, `br_b`, `nash`, `b_premier` et `candidate`. `candidate` liste les actions strictement inférieures de A contre Attaque. Pour B-premier, les égalités de A sont départagées en faveur de B, puis toutes les issues maximales de B sont rendues. Le contrôle utilise aussi une matrice modifiée et une égalité : une réponse constante ne suffit pas.

```python
# TODO etudiant
# Etape 1 : coder la matrice U_EXO1 et les correspondances br_a / br_b.
# Etape 2 : énumérer tous les équilibres de Nash.
# Etape 3 : induction à rebours B-premier et action candidate.
```


In [4]:
# EXERCICE 1 : analyser toutes les meilleures réponses, puis B-premier.
# Indice : énumérer les cases, conserver les égalités plutôt qu’un seul argmax.
U_EXO1 = {('PasCampagne','Reste'):(3,3), ('PasCampagne','Attaque'):(1,4),
          ('Campagne','Reste'):(2,0), ('Campagne','Attaque'):(0,-1)}
AA, BB = ['PasCampagne','Campagne'], ['Reste','Attaque']
def analyser_duel(matrice):
    # TODO étudiant : dictionnaire br_a, br_b, nash, b_premier, candidate.
    return None

def reference_b_premier(matrice):
    reponses = {b:max(AA, key=lambda a:(matrice[a,b][0],matrice[a,b][1])) for b in BB}
    meilleur = max(matrice[reponses[b],b][1] for b in BB)
    return [(reponses[b],b) for b in BB if matrice[reponses[b],b][1] == meilleur]

fixtures_duel = [U_EXO1, {k:(v[0]+(3 if k==('Campagne','Attaque') else 0),v[1]) for k,v in U_EXO1.items()},
                 {k:(v[0],0) for k,v in U_EXO1.items()}]
for matrice in fixtures_duel:
    r = analyser_duel(dict(matrice))
    if r is None:
        print('Exercice 1 à compléter : matrice, BR, Nash et B-premier')
        break
    assert set(r)=={'br_a','br_b','nash','b_premier','candidate'}
    ba = {b:{a for a in AA if matrice[a,b][0]==max(matrice[x,b][0] for x in AA)} for b in BB}
    bb = {a:{b for b in BB if matrice[a,b][1]==max(matrice[a,y][1] for y in BB)} for a in AA}
    assert set(r['br_a'])==set(BB) and set(r['br_b'])==set(AA)
    assert all(set(r['br_a'][b])==ba[b] for b in BB) and all(set(r['br_b'][a])==bb[a] for a in AA)
    assert set(map(tuple,r['nash']))=={(a,b) for a in AA for b in BB if a in ba[b] and b in bb[a]}
    assert set(map(tuple,r['b_premier']))==set(reference_b_premier(matrice))
    assert set(r['candidate'])=={a for a in AA if matrice[a,'Attaque'][0]<max(matrice[x,'Attaque'][0] for x in AA)}
else:
    print('Exercice 1 : BR, Nash, B-premier et action candidate validés sur trois matrices')


Exercice 1 à compléter : matrice, BR, Nash et B-premier


## Annonce révocable : induction dans les sous-arbres

L’annonce n’enlève ici aucune action au dernier décideur. Calculons sa meilleure réponse puis celle du premier joueur.

In [5]:
# === Regime 4 : l'annonce REVOCABLE -- l'engagement sans le lien ===
# Arbre a trois etages : I annonce (sans effet lie), puis E choisit, puis I choisit LIBREMENT.
# Toutes les annonces menent au meme sous-arbre : seule compte la decision finale de I.
def sous_arbre_final(ae):
    # Au dernier etage, I choisit librement entre Acc et Fight.
    return {ai: U[(ae, ai)] for ai in ACTIONS_I}

def induction_rebours(arbre_bas):
    # Etage du bas : I (indice 1) choisit dans {Acc: ..., Fight: ...}
    bas = {ae: max(sous.items(), key=lambda kv: kv[1][1]) for ae, sous in arbre_bas.items()}
    # Etage du haut : E (indice 0) choisit In / Out en anticipant bas
    issue = max(bas.items(), key=lambda kv: kv[1][1][0])
    return issue  # (action_E, (action_I, (u_E, u_I)))

arbre = {ae: sous_arbre_final(ae) for ae in ACTIONS_E}
action_E, (action_I, payoffs) = induction_rebours(arbre)
print('Annonce revocable (aucun lien), induction a rebours sur tout l arbre :')
for ae in ACTIONS_E:
    rep_I = max(ACTIONS_I, key=lambda ai: u_I(ae, ai))
    print(f'  si E joue {ae:4} : I choisit en dernier {rep_I:5} -> {U[(ae, rep_I)]}')
print(f'  -> issue : ({action_E}, {action_I})  ->  u_E = {payoffs[0]}, u_I = {payoffs[1]}')
print(f'  -> le gain d engagement (3) retombe a {payoffs[1]} : dissous, quelle que soit l annonce.')

Annonce revocable (aucun lien), induction a rebours sur tout l arbre :
  si E joue In   : I choisit en dernier Acc   -> (2, 1)
  si E joue Out  : I choisit en dernier Fight -> (0, 3)
  -> issue : (In, Acc)  ->  u_E = 2, u_I = 1
  -> le gain d engagement (3) retombe a 1 : dissous, quelle que soit l annonce.


### Lecture : la condition de félicité — l'annonce n'est pas l'engagement

Retirez le lien (l'irrévocabilité) et tout s'effondre dans l'ordre exact prédit par la théorie : au dernier étage, l'incumbent face à une entrée **réelle** accommode (1 > −2) — au moment de choisir, la menace n'a plus de valeur ; l'entrant, qui fait cette induction à rebours autant que nous, entre (2 > 0) ; et le gain de l'engagement **3 → 1**. Aucune annonce, si forte soit-elle, n'y change rien : l'issue est **indépendante de l'annonce**. Cette indépendance de l’annonce est calculée dans ce jeu à information commune et aux sous-arbres identiques ; elle n’est pas un théorème général d’inutilité de toute communication.

C'est ici que la condition de félicité d'Austin cesse d'être une métaphore littéraire. « Je combattrai » n'est pas un énoncé performatif en soi ; il ne le devient que si l'énonciation **modifie les gains ou les contraintes du locuteur** — si se dédire coûte. L'annonce révocable est un énoncé *constatif* déguisé : elle décrit une intention que tout le monde sait révisable, et l'anticipation collective la traite comme telle. La performativité n'est pas dans les mots : elle est dans **l'architecture du jeu que les mots modifient**.

## Caution : pénaliser la déviation

Le contrat retire s au gain d’Acc après une promesse Fight. Le balayage garde les neuf montants de 0 à 4, par pas de 0,5.

In [6]:
# === La credibilite au prix exact : le seuil de la caution (bond) ===
# L'incumbent signe un contrat : accommoder APRES avoir promis de combattre coute s (penalite).
# Payoff modifie : u_I(In, Acc) = 1 - s. Le reste est inchange.
def equilibre_avec_bond(s):
    # Dernier etage : I face a In compare Acc (1-s) a Fight (-2) ; face a Out : Fight (3 > 0).
    rep_in = 'Acc' if 1 - s > -2 else 'Fight'
    rep_out = 'Fight'
    # L'entrant anticipe :
    if rep_in == 'Fight':
        ae = 'Out'  # entrer ne rapporte que -1
        return (ae, 'Fight'), U[('Out', 'Fight')]
    ae = 'In'
    return (ae, rep_in), (2, 1 - s)

print(f'{"s (caution)":>12} {"issue":>16} {"u_I":>8}  regime')
for k in range(9):
    s = k * 0.5
    issue, pay = equilibre_avec_bond(s)
    regime = 'annonce dissoute (non credible)' if issue[1] == 'Acc' else 'dissuasion tenue (credible)'
    print(f'{s:>12.1f} {str(issue):>16} {pay[1]:>8.2f}  {regime}')
print()
_, pay_3 = equilibre_avec_bond(3.0)
ecart_tentation = u_I('In', 'Acc') - u_I('In', 'Fight')
print(f'Seuil de credibilite : s* = {ecart_tentation} (ecart de tentation 1 - (-2)).')
print(f'Au seuil s = 3 : u_I = {pay_3[1]} -- la pleine valeur d engagement est restauree.')

 s (caution)            issue      u_I  regime
         0.0    ('In', 'Acc')     1.00  annonce dissoute (non credible)
         0.5    ('In', 'Acc')     0.50  annonce dissoute (non credible)
         1.0    ('In', 'Acc')     0.00  annonce dissoute (non credible)
         1.5    ('In', 'Acc')    -0.50  annonce dissoute (non credible)
         2.0    ('In', 'Acc')    -1.00  annonce dissoute (non credible)
         2.5    ('In', 'Acc')    -1.50  annonce dissoute (non credible)
         3.0 ('Out', 'Fight')     3.00  dissuasion tenue (credible)
         3.5 ('Out', 'Fight')     3.00  dissuasion tenue (credible)
         4.0 ('Out', 'Fight')     3.00  dissuasion tenue (credible)

Seuil de credibilite : s* = 3 (ecart de tentation 1 - (-2)).
Au seuil s = 3 : u_I = 3 -- la pleine valeur d engagement est restauree.


### Lecture : le prix exact de la félicité — la caution doit couvrir la tentation

La caution transforme l'architecture : accommoder après avoir promis coûte désormais `1 − s`. Le balayage le montre avec une convention explicite à l’égalité : en dessous de `s* = 3`, l'entrant entre, l'incumbent accommode en payant la peine, et son gain **décroît de 1 − s** — le tableau le mesure ligne à ligne (0,50 à s = 0,5 ; 0,00 à s = 1 ; −1,50 à s = 2,5) : une caution sous le seuil est un coût pur, versée sans jamais acheter la dissuasion ; à partir de `s* = 3` (égalité exacte au seuil, stricte au-dessus), combattre devient la meilleure réponse **même face à une entrée réelle**, l'entrant le sait, reste dehors, et la valeur d'engagement **3** est intégralement restaurée. La condition de félicité d'Austin est devenue un nombre.

Et ce nombre a une forme remarquable : `s*` est exactement **l'écart de tentation** — la différence entre céder (1) et tenir (−2) au moment de la tester. Ce n'est ni le **delta** d'engagement (+2, mesuré la section correspondante : 3 avec engagement contre 1 sans engagement), ni le **dommage** de la guerre (2) : dans ce jeu, la caution à verser (3) **égale** la valeur d'engagement (3) — la « pleine valeur d'engagement » que la section correspondante restaure à `s = 3` — et **dépasse** le dommage de la guerre (2). Elle doit couvrir précisément ce que coûte le fait de dévier *au moment où dévier tenterait*. C'est la définition opérationnelle de la crédibilité : **crédible = dévier ne rapporte plus**, et le prix à payer pour y arriver se calcule sur la matrice. La « réputation », la « détermination », les métaphores volontaristes de la dissuasion se réduisent ici à cette inégalité.

À s = 3, Acc et Fight sont deux meilleures réponses après In. Le code sélectionne Fight au seuil ; une sélection Acc maintiendrait l’entrée. La dissuasion est strictement robuste pour s > 3. La pénalité change les gains ; ni trésorerie immobilisée ni prix financier de dépôt ne sont modélisés.


<a id="engagement-mixte"></a>

## Exercice 2 : le couple (engagement, gain) sur VOTRE matrice

Reprenez le duel publicitaire de l'Exercice 1. Construisez le tableau de l'engagement contraignant : pour **chacune** des deux actions pures de A, calculez la meilleure réponse de B et le gain de A qui en résulte. Produisez le **couple** `(stratégie d'engagement, gain)` optimal, et comparez-le au gain de A à l'issue séquentielle sans engagement — le delta d'engagement en un chiffre. Vérifiez enfin le mixte sur une grille `p ∈ [0, 1]` (pas de 0,05) : le sommet est-il pur, comme dans le jeu d'entrée, ou intérieur ?

La fonction étudiante retourne 21 entrées de la grille expérimentale (k = 0…20, p=k/20), chacune avec `k`, l’ensemble `br` des meilleures réponses de B, et les gains de A `optimiste`/`pessimiste`. Conserver les deux réponses quand B est indifférent. Le harnais vérifie chaque ligne sur plusieurs matrices puis calcule les meilleurs gains purs et mixtes, et leurs deltas contre B-premier. Les égalités du leader sont départagées par le premier ordre déclaré pour cette comparaison de référence.

```python
# TODO etudiant
# Etape 1 : pour chaque action de A, BR de B puis gain de A.
# Etape 2 : argmax : couple engagement/gain et delta.
# Etape 3 : balayage mixte p par pas de 0.05, égalités et sommet intérieur.
```


In [7]:
# EXERCICE 2 : tableau réel des 21 engagements, égalités incluses.
# Etape 1 : espérances par colonne ; Etape 2 : correspondance BR ; Etape 3 : les deux conventions.
def engagements_duel(matrice):
    # TODO étudiant : liste de dicts k, br, optimiste, pessimiste.
    return None

fixtures_mixtes = [U_EXO1, {k:(v[0],v[1]+(2 if k==('Campagne','Reste') else 0)) for k,v in U_EXO1.items()}]
for matrice in fixtures_mixtes:
    r = engagements_duel(dict(matrice))
    if r is None:
        print('Exercice 2 à compléter : engagements purs, mixte, deltas')
        break
    assert len(r)==21 and {ligne['k'] for ligne in r}==set(range(21))
    for ligne in r:
        assert set(ligne)=={'k','br','optimiste','pessimiste'}
        k=ligne['k']; assert isinstance(k,int) and not isinstance(k,bool) and 0<=k<=20
        # Calcul entier du numérateur : l’égalité à p=1/2 reste exacte.
        vb={b:(20-k)*matrice[AA[0],b][1]+k*matrice[AA[1],b][1] for b in BB}
        br={b for b in BB if vb[b]==max(vb.values())}
        va=[((20-k)*matrice[AA[0],b][0]+k*matrice[AA[1],b][0])/20 for b in br]
        assert set(ligne['br'])==br
        for key,attendu in [('optimiste',max(va)),('pessimiste',min(va))]:
            assert isinstance(ligne[key],(int,float)) and math.isfinite(ligne[key]) and abs(ligne[key]-attendu)<1e-10
    pur=max(x['optimiste'] for x in r if x['k'] in (0,20));mix=max(x['optimiste'] for x in r)
    base_issue=reference_b_premier(matrice)[0];base=matrice[base_issue][0]
    print(f'Exercice 2 : meilleur pur={pur:g}, meilleur mixte favorable={mix:g}, deltas={pur-base:g}/{mix-base:g}')


Exercice 2 à compléter : engagements purs, mixte, deltas


## Le témoin de retenue : le dual du témoin d'obstruction

Les régimes précédents définissent l'agence par ce qu'un joueur **peut** faire : s'engager,
annoncer, cautionner. Il manque le revers : **un coup profitable à l'instant présent est
accessible, et le joueur ne le joue pas**. Cette comparaison locale ne suffit pas à établir
que le coup améliorerait son gain sur tout l'horizon :

```
pouvoir disponible  ≠  pouvoir exercé
```

Le piège : une abstention observée admet **trois causes indiscernables sur la seule trajectoire** —
la retenue locale (l'objet mesuré ici), un barème erroné (la déviation *croyait-on* profitable),
l'incapacité (l'action n'était pas atteignable). Un détecteur qui ne sépare pas les trois certifie
de la retenue partout où il y a une erreur de modèle. Le témoin ne vaut donc que s'il produit
d'abord le **certificat de disponibilité locale** : le coup énuméré dans l'ensemble d'actions
ET strictement améliorant pour **une période** au barème courant — *calculé et affiché* —
avant de constater qu'il n'a pas été joué.

Le dispositif : le duopole accommodant de la section 1, répété. L'entrant peut choisir une
guerre de prix éphémère (un « hit-and-run »), profitable sur la période du raid mais qui ferme
ensuite le marché pour lui. Son abstention atteste qu'il n'exerce pas ce pouvoir disponible ;
elle ne prouve pas un sacrifice de gain total. Le verdict se lit **depuis la trajectoire** —
l'historique des actions — jamais depuis une utilité déclarée.

In [8]:
# === Section retenue A : le certificat de disponibilite — le coup hit-and-run ===

# Le duopole accommode : (In, Acc) chaque periode -> (2, 1) par periode.
# L'entrant possede une action supplementaire, ephemere : 'Raider' (guerre de prix UNE periode,
# puis le marche se ferme a jamais pour lui). Convention : (u_entrant, u_incumbent).
GAINS_DUOPOLE = {('In', 'Acc'): (2, 1), ('Raider', 'Acc'): (5, -3), ('Out', 'Acc'): (0, 0)}
HORIZON = 10  # periodes de duopole accommodant

def gain_entrant(histoire):
    """Gain total de l'entrant LU DEPUIS LA TRAJECTOIRE (suite d'actions, pas d'utilites)."""
    total = 0
    for ae, ai in histoire:
        total += GAINS_DUOPOLE[(ae, ai)][0]
    return total

def actions_atteignables():
    return list(GAINS_DUOPOLE.keys())

# --- Le certificat de disponibilite, calcule et affiche ---
actions = actions_atteignables()
coup_raider = ('Raider', 'Acc')
gain_raider_1p = GAINS_DUOPOLE[coup_raider][0]
gain_in_1p = GAINS_DUOPOLE[('In', 'Acc')][0]

print("Certificat de disponibilite du coup 'Raider' pour l'entrant :")
print(f"  1. ENUMERATION : 'Raider' figure dans l'ensemble d'actions atteignables = {actions}")
print(f"  2. GAIN STRICT : raider une periode rapporte {gain_raider_1p} > cooperer {gain_in_1p}"
      f" (delta immediat = +{gain_raider_1p - gain_in_1p})")
print(f"  3. La deviation est donc DISPONIBLE et STRICTEMENT PROFITABLE au bareme courant.")
print()
print("Le certificat pose, l'experience peut commencer : que fait l'entrant ?")

Certificat de disponibilite du coup 'Raider' pour l'entrant :
  1. ENUMERATION : 'Raider' figure dans l'ensemble d'actions atteignables = [('In', 'Acc'), ('Raider', 'Acc'), ('Out', 'Acc')]
  2. GAIN STRICT : raider une periode rapporte 5 > cooperer 2 (delta immediat = +3)
  3. La deviation est donc DISPONIBLE et STRICTEMENT PROFITABLE au bareme courant.

Le certificat pose, l'experience peut commencer : que fait l'entrant ?


**Lecture de la sortie committée** : le certificat tient en deux lignes vérifiables — l'action
est dans l'ensemble énuméré, et son gain **immédiat** (5 > 2) est calculé au barème courant,
pas déclaré. Il établit une occasion profitable sur une période, pas un gain supérieur sur les
dix périodes du jeu répété. La trajectoire suivante permet de mesurer séparément cet effet
à long terme.

In [9]:
# === Section retenue B : deux trajectoires sur le MEME jeu — exploit vs abstention ===

def trajectoire_exploit():
    """L'entrant raide a la periode 3, puis le marche se ferme pour lui (Out force)."""
    h = [('In', 'Acc')] * 2 + [('Raider', 'Acc')] + [('Out', 'Acc')] * (HORIZON - 3)
    return h

def trajectoire_retenue():
    """L'entrant NE joue PAS le coup disponible : duopole accommodant sur tout l'horizon."""
    h = [('In', 'Acc')] * HORIZON
    return h

def lire_raider_depuis_trajectoire(h):
    """Detecteur : le coup Raider apparaît-il dans l'historique des ACTIONS ?"""
    return any(ae == 'Raider' for ae, _ in h)

for nom, h in [("trajectoire EXPLOIT", trajectoire_exploit()),
               ("trajectoire RETENUE", trajectoire_retenue())]:
    lu = lire_raider_depuis_trajectoire(h)
    print(f"{nom:22s} : raider observe dans la trajectoire = {lu}")
    print(f"{'':22s}   gain total de l'entrant (lu, non declare) = {gain_entrant(h)}")
    print(f"{'':22s}   actions distinctes jouees = {sorted(set(ae for ae, _ in h))}")
print()
print("Divergence des deux histoires : MEME jeu, MEME certificat de disponibilite,")
print("deux pouvoirs exercés différents — la retenue est le second.")

trajectoire EXPLOIT    : raider observe dans la trajectoire = True
                         gain total de l'entrant (lu, non declare) = 9
                         actions distinctes jouees = ['In', 'Out', 'Raider']
trajectoire RETENUE    : raider observe dans la trajectoire = False
                         gain total de l'entrant (lu, non declare) = 20
                         actions distinctes jouees = ['In']

Divergence des deux histoires : MEME jeu, MEME certificat de disponibilite,
deux pouvoirs exercés différents — la retenue est le second.


**Lecture de la sortie committée** : les deux trajectoires divergent sur un fait lisible —
la présence ou l'absence du raid dans **l'historique des actions**. Le gain total de l'entrant
est *calculé depuis* cet historique (jamais demandé au joueur) : l'exploit rapporte **9**
(deux périodes de duopole à 2, un raid à 5, puis sept périodes dehors à 0), contre **20**
(dix périodes à 2) lorsqu'il s'abstient. L'exploit rapporte donc **moins**, et non plus, sur
l'horizon de dix périodes. Le certificat précédent prouve seulement un avantage **immédiat**
(5 > 2 sur la période du raid) ; il ne certifie pas un avantage cumulé une fois la fermeture
du marché prise en compte. Ce témoin distingue un coup disponible non exercé d'un coup exercé,
mais ne démontre pas que l'abstention est sous-optimale sur l'horizon complet.

In [10]:
# === Section retenue C : les trois controles negatifs — separer les causes d'abstention ===

def certificat_disponibilite(action, bareme, br_actuelle, gain_actuel):
    """Un coup n'est DISPONIBLE que s'il est atteignable ET strictement ameliorant au bareme."""
    atteignable = action in bareme
    if not atteignable:
        return False, "INCAPACITE : l'action n'est pas dans l'ensemble enumere"
    delta = bareme[action][0] - gain_actuel
    if delta <= 0:
        return False, f"BAREME : le coup rapporte {bareme[action][0]} <= {gain_actuel} (non profitable)"
    return True, f"DISPONIBLE : enumere, +{delta} strictement ameliorant"

CAS = [
    # (nom, action_testee, bareme, gain_actuel, histoire_observee)
    ("vraie retenue", ('Raider', 'Acc'), GAINS_DUOPOLE, 2, trajectoire_retenue()),
    ("abstention par BAREME errone", ('Raider', 'Acc'), {('In', 'Acc'): (2, 1), ('Raider', 'Acc'): (2, -3), ('Out', 'Acc'): (0, 0)}, 2, trajectoire_retenue()),
    ("abstention par INCAPACITE", ('Raider', 'Acc'), {('In', 'Acc'): (2, 1), ('Out', 'Acc'): (0, 0)}, 2, trajectoire_retenue()),
]

verdicts = []
for nom, action, bareme, gain_actuel, histoire in CAS:
    abstention = not lire_raider_depuis_trajectoire(histoire)
    dispo, motif = certificat_disponibilite(action, bareme, None, gain_actuel)
    if abstention and dispo:
        verdict = "RETENUE (abstention + coup disponible)"
    elif abstention and not dispo:
        verdict = f"ABSTENTION EXPLIQUEE SANS RETENUE : {motif}"
    else:
        verdict = "COUP JOUE (pas une abstention)"
    verdicts.append((nom, verdict, motif))
    print(f"--- {nom}")
    print(f"    bareme de 'Raider' : {bareme.get(action, 'ABSENT de l ensemble')}")
    print(f"    certificat : {motif}")
    print(f"    verdict    : {verdict}")

print()
uniques = set(v for _, v, _ in verdicts)
print(f"Verdicts distincts sur les trois cas : {len(uniques)}/3")
print("Un detecteur qui rendait le meme verdict trois fois serait rejete (acceptance #3).")

--- vraie retenue
    bareme de 'Raider' : (5, -3)
    certificat : DISPONIBLE : enumere, +3 strictement ameliorant
    verdict    : RETENUE (abstention + coup disponible)
--- abstention par BAREME errone
    bareme de 'Raider' : (2, -3)
    certificat : BAREME : le coup rapporte 2 <= 2 (non profitable)
    verdict    : ABSTENTION EXPLIQUEE SANS RETENUE : BAREME : le coup rapporte 2 <= 2 (non profitable)
--- abstention par INCAPACITE
    bareme de 'Raider' : ABSENT de l ensemble
    certificat : INCAPACITE : l'action n'est pas dans l'ensemble enumere
    verdict    : ABSTENTION EXPLIQUEE SANS RETENUE : INCAPACITE : l'action n'est pas dans l'ensemble enumere

Verdicts distincts sur les trois cas : 3/3
Un detecteur qui rendait le meme verdict trois fois serait rejete (acceptance #3).


**Lecture de la sortie committée** : les trois cas rendent **trois verdicts distincts** — le
certificat sépare l'abstention malgré un avantage **immédiat**, le barème sans avantage immédiat
et l'incapacité. La « vraie retenue » dans ce contrôle exige la conjonction *abstention observée*
+ *coup énuméré* + *gain d'une période strictement supérieur au barème*. Ce verdict local ne
tranche pas si le coup aurait accru le gain cumulé : dans les trajectoires précédentes, le raid
fait précisément baisser celui-ci de 20 à 9. La caution de la section précédente rendait
l'annonce contraignante au prix exact ; ici, le certificat rend l'abstention significative à
l'échelle d'une décision, sans assimiler les deux échelles de gain.

**Ce que ce témoin n'est pas** : pas un score de coopération (coopérer sans avantage immédiat
ne satisfait pas ce certificat — voir le contrôle « barème » ; une action inaccessible échoue
au contrôle « incapacité ») ; pas une norme morale encodée, ni la preuve d'un sacrifice sur dix
périodes. Il mesure une action accessible non exercée, distincte de l'optimisation globale que
la sortie de trajectoire favorise ici.

<a id="annonce-caution"></a>

## Exercice 3 : l'évanouissement mesuré — annonce révocable et caution minimale

Toujours sur le duel publicitaire : (a) codez l'arbre à trois étages « A annonce, B choisit, A choisit librement » et montrez par induction à rebours que l'issue est **indépendante de l'annonce** — le gain d'engagement s'évanouit ; (b) introduisez la caution `s` qui pénalise l'écart de promesse de A, balayez `s` et trouvez le seuil `s*` où l'engagement redevient crédible ; (c) vérifiez que `s*` égale l'écart de tentation de A sur la colonne critique. Sans (c), la félicité n'a pas été mesurée.

La fonction étudiante reçoit une matrice et s≥0. Retourner `reponses` (toutes les meilleures réponses de A après chaque choix de B, avec pénalité pour PasCampagne dans **les deux colonnes**), `issues` (B-premier, tie de A favorable à la promesse Campagne), `annonces` (les deux annonces révocables sans pénalité) et `seuils` (écarts de tentation non négatifs dans chacune des colonnes). Le contrat pénalise toute déviation de Campagne ; pénaliser une seule case peut laisser une déviation profitable ailleurs. Le contrôle fait varier s et les gains. Au seuil les réponses peuvent être multiples ; au-dessus du maximum des deux écarts, Campagne devient strictement optimale dans les deux sous-jeux.

```python
# TODO etudiant
# Etape 1 : arbre à trois étages et induction à rebours.
# Etape 2 : caution s sur toute déviation, dans les deux colonnes.
# Etape 3 : s* contre les deux écarts de tentation ; ties au seuil.
```


In [11]:
# EXERCICE 3 : toute déviation PasCampagne est pénalisée dans LES DEUX colonnes.
# Indice : comparer les deux sous-jeux ; les seuils changent avec les gains.
def annonce_caution_duel(matrice, s):
    # TODO étudiant : reponses, issues, annonces (sans pénalité), seuils.
    return None

def verifier_caution_duel():
    autres={k:(v[0]+(3 if k==('PasCampagne','Reste') else 0),v[1]) for k,v in U_EXO1.items()}
    for matrice in [U_EXO1,autres]:
        annonces = {a:reference_b_premier(matrice) for a in AA}
        seuils={b:max(0,matrice[AA[0],b][0]-matrice[AA[1],b][0]) for b in BB}
        for s in (0,.5,1,1.5,2,4,4.5):
            r=annonce_caution_duel(dict(matrice),s)
            if r is None:
                print('Exercice 3 à compléter : arbre révocable et pénalité de toute déviation')
                return
            assert set(r)=={'reponses','issues','annonces','seuils'}
            gains={(a,b):matrice[a,b][0]-(s if a==AA[0] else 0) for a in AA for b in BB}
            reponses={b:{a for a in AA if gains[a,b]==max(gains[x,b] for x in AA)} for b in BB}
            assert set(r['reponses'])==set(BB) and all(set(r['reponses'][b])==reponses[b] for b in BB)
            choisis={b:max(reponses[b],key=lambda a:a=='Campagne') for b in BB}
            val=max(matrice[choisis[b],b][1] for b in BB)
            issues={(choisis[b],b) for b in BB if matrice[choisis[b],b][1]==val}
            assert set(map(tuple,r['issues']))==issues and set(r['annonces'])==set(AA)
            assert all(set(map(tuple,r['annonces'][a]))==set(annonces[a]) for a in AA)
            assert set(r['seuils'])==set(BB) and all(isinstance(r['seuils'][b],(int,float)) and math.isfinite(r['seuils'][b]) and abs(r['seuils'][b]-seuils[b])<1e-10 for b in BB)
    print('Exercice 3 : deux matrices, quatorze sous-jeux, ties et seuils validés')
verifier_caution_duel()


Exercice 3 à compléter : arbre révocable et pénalité de toute déviation


## Résumé : quatre régimes, un seul nombre qui change, et ce qu'il prouve

| régime | mécanisme | issue | u_I |
|---|---|---|---|
| simultané, équilibre crédible | Nash `(In, Acc)` | entrée accueillie | 1 |
| simultané, équilibre de menace | Nash `(Out, Fight)` — incroyable | dissuasion *non tenable* | (3, fantôme) |
| séquentiel sans engagement | induction à rebours | `(In, Acc)` — la menace dissoute | 1 |
| **engagement contraignant** | `sigma_L -> BR_F(sigma_L)` | **`(Out, Fight)` — sélectionné** | **3** |
| annonce révocable | cheap talk — arbre à 3 étages | `(In, Acc)` — indépendant de l'annonce | 1 |
| engagement + caution s > 3 (ou s=3 avec tie Fight) | dévier ne rapporte plus | `(Out, Fight)` tenu sans jamais combattre | 3 |

Le noyau tient en trois phrases. **Dans cette matrice, l’engagement sélectionne un équilibre existant** que la menace seule ne pouvait pas tenir — le delta +2 était déjà dans la matrice, endormi dans la colonne `Out`. **L'acte engagé vaut par la modification de l'espace de réponse d'autrui**, non par son résultat direct : `Fight` vaut −2 *exécutée* contre une entrée, et 3 *commise* contre l'entrée qui n'aura pas lieu. **La félicité se mesure** : dans cet arbre, une annonce sans lien ne change pas l’issue, et restaurer le lien coûte exactement l'écart de tentation `s* = 3` — la caution minimale telle que dévier ne rapporte plus.

C'est une performativité sans mystère au sens strict : la chaîne `représentation publique -> anticipation collective -> changement des actions -> nouvelle réalité stratégique` est ici un calcul réversible, chaque flèche étant une cellule de ce notebook. Ce que le cadre ne couvre **pas** — et il faut l'écrire — : les contextes où l'engagement est multi-décisionnel et répété (réputation dynamique), où la caution est endogène (capacités détruites plutôt que caution versée), et les jeux où un engagement *mixte* bat le sommet pur (déjà illustré par l’exercice du duel publicitaire). Les autres extensions demandent un modèle supplémentaire — pas une extrapolation sociologique de celui-ci.

**Ouvertures de la série** : la sélection d'équilibre par contrainte rejoint les [jeux répétés et le folk theorem](Dynamics/Dynamics-02-Jeux-Repetes-Lean-Python.ipynb) (l'engagement comme cas limite de la punition crédible), et la modification de l'espace des actions plutôt que du choix en leur sein est exactement le geste de la strate 7 sur les vocabulaires — voir aussi [Metajeux-01, les open games](Metajeux/Metajeux-01-Open-Games-Lentilles-Python.ipynb), où la représentation locale modifie le global dont elle est issue.

## Exercices — récapitulatif

Trois activités restent ouvertes : [duel publicitaire](#duel-publicitaire), [engagement pur et mixte](#engagement-mixte), et [annonce/caution](#annonce-caution). Les contrôles n’affichent une validation qu’après une réponse fournie.

## À retenir

| Question | Objet calculé | Limite |
|---|---|---|
| Menace ou engagement ? | BR après choix contraignant | Information commune dans ce modèle |
| Annonce crédible ? | Gains de déviation et seuil de pénalité | Tie explicite au seuil |
| Retenue locale ? | Action accessible, gain immédiat, historique | Pas une preuve de sacrifice cumulé |

## Vérifiez votre compréhension

<details><summary>Pourquoi Fight n’est-elle pas globalement dominée ?</summary>Elle rapporte 3 contre Out, davantage que Acc ; son infériorité ne concerne que In.</details>

<details><summary>Pourquoi le seuil de caution n’est-il pas le delta d’engagement ?</summary>Le seuil compare tenir/dévier dans un sous-jeu, le delta compare deux mécanismes initiaux.</details>

<details><summary>Pourquoi l’abstention locale ne prouve-t-elle pas un sacrifice sur dix périodes ?</summary>Le raid rapporte 9 sur l’horizon contre 20 sans raid, malgré son avantage immédiat 5 contre 2.</details>

## Pour aller plus loin

Von Stackelberg (1934), *Marktform und Gleichgewicht* ; Austin (1962), *How to Do Things with Words*. Le [carnet suivant](GameTheory-09c-Stackelberg-SecurityGame-Python.ipynb) étudie les engagements de sécurité. Les liens de la série servent de points d’entrée ; aucune réplication empirique n’est promise ici.
